# Bronze - ATP Matches

- This process consists of aggregation all new data extracted daily from website https://stats.tennismylife.org/tennis-match-database 
- Only new items will be append to the final dataset

## Imports

In [9]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f

import pandas as pd

import os
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

from dotenv import load_dotenv
load_dotenv()

True

## Init spark

In [10]:
try:
    spark = SparkSession.builder.appName("bronze").getOrCreate()
except Exception as e:
    print(e)

## Load database

In [11]:
tb_atp_matches = (
    spark.read
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "bronze.tb_atp_matches")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .load()
    )

In [12]:
tb_ongoing_tourneys = spark.read.format("csv").option("header", "true").load(r"../../data/raw/daily/tb_ongoing_tourneys.csv")

## Append new data

In [13]:
new_matches = (
    tb_atp_matches.alias("tb_matches")
    .join(
        tb_ongoing_tourneys.alias("tb_ongoing"),
        [
            f.col("tb_matches.tourney_date") == f.col("tb_ongoing.tourney_date"), 
            f.col("tb_matches.winner_name") == f.col("tb_ongoing.winner_name")
        ],
        'right'
        )
    .where(f.col("tb_matches.tourney_date").isNull())
    .select(
        "tb_ongoing.*"
    )
)

In [14]:
df = tb_atp_matches.unionByName(new_matches, allowMissingColumns=True)

## Save dataframe

### Local

In [15]:
df.toPandas().to_csv(
    r"../../data/bronze/tb_atp_matches.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)

### Supabase

In [16]:
(
new_matches.write
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "bronze.tb_atp_matches")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .mode("append")
    .save()
)